# Droplet in a Shear Flow

This tutorial illustrates how to use the 
incompressible multiphase solver (`XNSE`),
computing a droplet in a shear-flow configuration.


First, we initialize the new worksheet;
Note: 
1. This tutorial can be found in the source code repository as `DropletInShearFlow.ipynb`. 
   One can directly load this into Jupyter to interactively work with the following code examples.
2. **In the following line, the reference to `BoSSSpad.dll` is required**. 
   You must either set `#r "BoSSSpad.dll"` to something which is appropriate for your computer
   (e.g. `C:\Program Files (x86)\FDY\BoSSS\bin\Release\net8.0\BoSSSpad.dll` if you installed the binary distribution),
   or, if you are working with the source code, you must compile `BoSSSpad` and put it side-by-side to this worksheet file
   (from the original location in the repository, you can use the scripts `getbossspad.sh`, resp. `getbossspad.bat`).

In [ ]:
#r "BoSSSpad.dll"
//#r "../../../src/L4-application/BoSSSpad/bin/Debug/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();


## Batch Processing

First, we have to select a *batch system* (aka.*execution queue*, aka. *queue*) that we want to use.
Batch systems are a common approach to organize workloads (aka. compute jobs)
on compute clusters.
On such systems, a user typically does **not** starts a simulation manually/interactively.
Instead, he specifies a so-called *compute job*. The *scheduler* 
(i.e. the batch system) collects 
compute jobs from all users on the compute cluster, sorts them according to 
some priority and puts the jobs into some queue, also called *batch*.
The jobs in the batch are then executed in order, depending on the 
available hardware and the scheduling policies of the system.

The *BoSSS* API provides front-ends (clients) for the following 
batch system software:

- `BoSSS.Application.BoSSSpad.SlurmClient` for the 
Slurm Workload Manager (very prominent on Linux HPC systems)
- `BoSSS.Application.BoSSSpad.MsHPC2012Client`
for the Microsoft HPC Pack 2012 and higher
- `BoSSS.Application.BoSSSpad.MiniBatchProcessorClient` for the 
mini batch processor, a minimalistic, *BoSSS*-internal batch system which mimiks 
a supercomputer batch system on the local machine.


A list of clients for various batch systems, which are loaded at the 
`Init()` command can be configured through the  
`~/.BoSSS/etc/BatchProcessorConfig.json`-file.
If this file is missing, a default setting, containing a 
mini batch processor, is initialized. 

The list of all execution queues can be accessed through:

In [ ]:
ExecutionQueues

In order to run a simulation job, one can either manually select one of these queues -- or, one culd just use the **default queue**.  The default queue for execution can be configured by two options:
- globally, can specified by the `DefaultQueueIndex` in configuration file `~/.BoSSS/etc/BatchProcessorConfig.json`
- for each project (see below), it can be overwritten using the file `~/.BoSSS/etc/DefaultQueuesProjectOverride.txt`


## Initializing the workflow management

In order to use the workflow management, 
the very first thing we have to do is to initialize it by defineing 
a **project name**, here it is `MetaJobManager_Tutorial`. 
This is used to generate names for the compute jobs and to 
identify sessions in the database:

In [ ]:
BoSSSshell.WorkflowMgm.Init("DropletInShearFlow");

In [ ]:
// tell the workflow management to only look at the 'session name' for comparing simulations
// (alternatively, all members of a control object are used for comparison)
wmg.SetNameBasedSessionJobControlCorrelation();

For this project, the default execution queue is set to:

In [ ]:
GetDefaultQueue()

We verify that we have no jobs defined so far ...

In [ ]:
BoSSSshell.WorkflowMgm.AllJobs

In [ ]:
// the folloowing line is part of the trest system and not neccesary in User worksheets:
NUnit.Framework.Assert.IsTrue(BoSSSshell.WorkflowMgm.AllJobs.Count == 0, "MetaJobManager tutorial: expecting 0 jobs on entry.");

The initialization of the Workflow Management environment already creates, resp. opens
a *BoSSS database* with the same name as the project name as the project. The current default database is set as:

In [ ]:
wmg.DefaultDatabase

### Notes on databases:
* For expensive simulations, which run for days or longer, temporaray databases,
  which are created with a different name every time the notebook is executed, 
  are typically **not** desired.
  Hence, one wants the compute jobs to persist, i.e. if the worksheet is re-executed (maybe on another day), 
  but the computation 
  has been successful somewhen in the past, this result is recovered from the database.
  In such a scenario, one cannot use a temporary database, but the default project database
  should be used instead.
* Under certain circumstances, one could also create additional databases:
  Using the methods `BatchProcessorClient.CreateTempDatabase()`, 
  resp. `BatchProcessorClinet.CreateOrOpenCompatibleDatabase(...)`.
  (as demonstrated below) ensures that the database is in a directory which can be accessed by the batch system.
  (Alternative functions, i.e. `BoSSSshell.CreateTempDatabase()` or `BoSSSshell.OpenOrCreateDatabase(...)` 
  do not guarantee this and the user has to ensure an appropriate location.

All currently opened databases can be listed using:

In [ ]:
databases

## Setup of the Test-Case

Initially, a circular droplet of radius of 0.5 is laced in the center of a domain $(-1,1) \times (-1,1)$.
The shear flow is driven by the top- and bottom walls, which induce a tangential velocity of $\vec{u}(y=1) = (s,0)$, resp., $\vec{u}(y=-1) = (-s,0)$,
where $s$ is the shear rate.
On the left and right side of the domain, outflow boundary conditions apply.

All constants, i.e., densities and viscosities of both fluids, and surface tension´, are set to 1.0.

In [ ]:
using BoSSS.Application.XNSE_Solver;

### Grid with boundary conditions:

In [ ]:
int Res = 50;
var xNodes       = GenericBlas.Linspace(-1, 1, Res + 1); 
var yNodes       = GenericBlas.Linspace(-1, 1, Res + 1); 
GridCommons grid = Grid2D.Cartesian2DGrid(xNodes, yNodes);
grid.DefineEdgeTags(delegate (double[] X) { 
    double x = X[0];
    double y = X[1]; 
    if (Math.Abs(y - (-1)) <= 1.0e-8) 
        return "wall_bottom"; // lower wall
    if (Math.Abs(y - (+1)) <= 1.0e-8) 
        return "wall_top"; // upper wall
    if (Math.Abs(x - (-1)) <= 1.0e-8) 
        return "Dong_OutFlow_left"; // left: on the upper part (y > 0), there will actually be inflow; 
                                    // The Dong-Boundary condition in more stable on "outflows" which might actually have inflow. 
    if (Math.Abs(x - (+1)) <= 1.0e-8) 
        return "Dong_OutFlow_right"; // outlet
    throw new ArgumentOutOfRangeException("unknown domain"); 
});

One can save this grid explicitly to a database, but it is not a must;
The grid should be saved automatically, when the job is activated.

In [ ]:
wmg.DefaultDatabase.SaveGrid(ref grid);
grid

### Simulation Setup:

In [ ]:
double shearRate = 0.1;
int dg_degree = 2;

In [ ]:
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;

In [ ]:
var c = new XNSE_Control();
// general description:
int k                = 2;
string desc          = "droplet in a shear flow, shear rate = " + shearRate + ", k" + k; 
c.SessionName        = $"ShearRate_{shearRate}"; 
c.ProjectDescription = desc;
c.savetodb           = true;
c.Tags.Add("k" + dg_degree);


// setting the grid:
c.SetGrid(grid);

// DG polynomial degree
c.SetDGdegree(k);

// Physical parameters:
c.PhysicalParameters.rho_A = 1.0; 
c.PhysicalParameters.mu_A  = 1.0;
c.PhysicalParameters.rho_B = 1.0; 
c.PhysicalParameters.mu_B = 1.0;
c.PhysicalParameters.Sigma = 1.0;

// Boundary conditions:
var WallVelocity = new Formula($"X => X[1]*{shearRate}", false); // 2nd Argument=false says that its a time-indep. formula.
c.AddBoundaryValue("wall_top", "VelocityX", WallVelocity);
c.AddBoundaryValue("wall_bottom", "VelocityX", WallVelocity);

// Initial droplet
var phi0 = new Formula($"X => 0.5 - X.L2Norm()", false); // 2nd Argument=false says that its a time-indep. formula.
c.AddInitialValue("Phi", phi0);


// Timestepping properties:
c.dtFixed = 0.05;
c.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;

// Experimental options which are supposed to change
c.StokesExtentionUseBCmap = true;
c.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
c.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
c.ReInitPeriod = 100;

In [ ]:
// assertions to ensure that no developer has messed with certain default values:
Assert.IsTrue(c.IncludeConvection, "convective terms should be activated per default");
Assert.IsTrue(c.IncludeDiffusion, "viscous terms should never be turned off");
Assert.Equals(c.TimesteppingMode, AppControl._TimesteppingMode.Transient, "setting a dt should automatically ensure that a transient simulation is set");

## Activation and Monitoring of the the Job

Finally, we are ready to deploy the job at the batch processor;
In a usual work flow scenario, we **do not** want to (re-) submit the 
job every time we run the worksheet -- usually, one wants to run a job once.

The concept to overcome this problem is job activation. If a job is 
activated, the meta job manager first checks the databases and the batch 
system, if a job with the respective name and project name is already 
submitted. Only if there is no information that the job was ever submitted
or started anywhere, the job is submitted to the respective batch system.

First, a `Job* -object is created from the control object:

In [ ]:
var JobLocal = c.CreateJob();

This job is not activated yet, it can still be configured:

In [ ]:
JobLocal.Status

In [ ]:
// Test:
NUnit.Framework.Assert.IsTrue(JobLocal.Status == JobStatus.PreActivation);

### Starting the compute Job

One can change e.g. the number of MPI processes:

In [ ]:
JobLocal.NumberOfMPIProcs = 2;
JobLocal.NumberOfThreads = 4;
JobLocal.Activate();  // execute the job in the default execution queue

Note that these jobs are desigend to be **persistent**:
This means the computation is only started 
**once for a given control object**, no matter how often the worksheet
is executed. 

Such a behaviour is useful for expensive simulations, which run on HPC
servers over days or even weeks. The user (you) can close the worksheet
and maybe open and execute it a few days later, and he can access
the original job which he submitted a few days ago (maybe it is finished
now).

Then, the job is activated, resp. submitted, resp. deployed 
to one batch system.
If job persistency is not wanted, traces of the job can be removed 
on request during activation, causing a fresh job deployment at the
batch system:

All jobs can be listed using the workflow management:

In [ ]:
BoSSSshell.WorkflowMgm.AllJobs

Check the present job status:

In [ ]:
JobLocal.Status

In [ ]:
/// BoSSScmdSilent BoSSSexeSilent
NUnit.Framework.Assert.IsTrue(
   JobLocal.Status == JobStatus.PendingInExecutionQueue
   || JobLocal.Status == JobStatus.InProgress
   || JobLocal.Status == JobStatus.FinishedSuccessful);

### Evaluation of Job

Here, we block until both of our jobs have finished:

In [ ]:
BoSSSshell.WorkflowMgm.BlockUntilAllJobsTerminate(36000);

We examine the output and error stream of the job:
This directly accesses the *\tt stdout*-redirection of the respective job
manager, which may contain a bit more information than the 
`Stdout`-copy in the session directory.

In [ ]:
JobLocal.Stdout

We can also obtain the session 
which was stored during the execution of the job:

In [ ]:
var Sloc = JobLocal.LatestSession;
Sloc